# Used Car Price Prediction

## 1) Problem statement.

* This dataset comprises used cars sold on cardehko.com in India as well as important features of these cars.
* If user can predict the price of the car based on input features.
* Prediction results can be used to give new seller the price suggestion based on market condition.

## 2) Data Collection.
* The Dataset is collected from scrapping from cardheko webiste
* The data consists of 13 column and 15411 rows.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import warnings

warnings.filterwarnings("ignore")

%matplotlib inline

In [3]:
df = pd.read_csv("cardekho_imputated.csv")
df.head()

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [4]:
df.isnull().sum()

Unnamed: 0           0
car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [5]:
df.drop(['car_name','brand'], axis=1, inplace=True)

In [6]:
df['model'].unique()

array(['Alto', 'Grand', 'i20', 'Ecosport', 'Wagon R', 'i10', 'Venue',
       'Swift', 'Verna', 'Duster', 'Cooper', 'Ciaz', 'C-Class', 'Innova',
       'Baleno', 'Swift Dzire', 'Vento', 'Creta', 'City', 'Bolero',
       'Fortuner', 'KWID', 'Amaze', 'Santro', 'XUV500', 'KUV100', 'Ignis',
       'RediGO', 'Scorpio', 'Marazzo', 'Aspire', 'Figo', 'Vitara',
       'Tiago', 'Polo', 'Seltos', 'Celerio', 'GO', '5', 'CR-V',
       'Endeavour', 'KUV', 'Jazz', '3', 'A4', 'Tigor', 'Ertiga', 'Safari',
       'Thar', 'Hexa', 'Rover', 'Eeco', 'A6', 'E-Class', 'Q7', 'Z4', '6',
       'XF', 'X5', 'Hector', 'Civic', 'D-Max', 'Cayenne', 'X1', 'Rapid',
       'Freestyle', 'Superb', 'Nexon', 'XUV300', 'Dzire VXI', 'S90',
       'WR-V', 'XL6', 'Triber', 'ES', 'Wrangler', 'Camry', 'Elantra',
       'Yaris', 'GL-Class', '7', 'S-Presso', 'Dzire LXI', 'Aura', 'XC',
       'Ghibli', 'Continental', 'CR', 'Kicks', 'S-Class', 'Tucson',
       'Harrier', 'X3', 'Octavia', 'Compass', 'CLS', 'redi-GO', 'Glanza',
       

In [8]:
num_features = len([features for features in df.columns if df[features].dtype != 'object'])
char_features = len([features for features in df.columns if df[features].dtype == 'object'])
discrete_features = len([features for features in df.columns if len(df[features].unique()) < 25])
continuous_features = len([features for features in df.columns if len(df[features].unique()) >= 25])
print(num_features, "numerical features")
print(char_features, "categorical features")
print(discrete_features, "discrete features")
print(continuous_features, "continuous features")

8 numerical features
4 categorical features
5 discrete features
7 continuous features


In [10]:
from sklearn.model_selection import train_test_split
X = df.drop('selling_price', axis=1)
y = df['selling_price']

In [11]:
X.head()

,Unnamed: 0,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


In [12]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
X['model'] = le.fit_transform(X['model'])

In [15]:
num_features = X.select_dtypes(exclude='object').columns
onehot_columns=['seller_type','fuel_type','transmission_type']

from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(drop='first')
preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", categorical_transformer, onehot_columns),
        ("StandardScaler", numeric_transformer, num_features)
    ],remainder='passthrough'
)

In [16]:
X = preprocessor.fit_transform(X)

In [17]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [19]:
X_train.shape , y_train.shape , X_test.shape , y_test.shape

((12328, 15), (12328,), (3083, 15), (3083,))

In [20]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from streamlit import form

from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error

In [21]:
def evaluate_model(true,predicted):
    mse = mean_squared_error(true,predicted)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(true,predicted)
    r2 = r2_score(true,predicted)
    
    return mse,rmse,mae,r2

In [ ]:
models ={
    "LinearRegression" : LinearRegression(),
    "DecisionTreeRegressor" : DecisionTreeRegressor(),
    "RandomForestRegressor" : RandomForestRegressor(),
    "KNeighborsRegressor" : KNeighborsRegressor()
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    
    mse,rmse,mae,r2 = evaluate_model(y_test,y_pred)
    
    print(f"Model : {list(models.keys())[i]}")
    print(f"MSE : {mse}")
    print(f"RMSE : {rmse}")
    print(f"MAE : {mae}")
    print(f"R2 Score : {r2}")
    print("\n")

Model : LinearRegression
MSE : 252588750547.228
RMSE : 502582.0833925817
MAE : 279686.6479172668
R2 Score : 0.6644595369594529


Model : DecisionTreeRegressor
MSE : 100797348123.78365
RMSE : 317485.98098779673
MAE : 128471.90236782355
R2 Score : 0.8661001775041848


Model : RandomForestRegressor
MSE : 57721280925.92442
RMSE : 240252.53573255875
MAE : 98245.48653908531
R2 Score : 0.923322692371619


Model : KNeighborsRegressor
MSE : 91424152395.59682
RMSE : 302364.27103015466
MAE : 124343.66688290625
R2 Score : 0.8785515888516486




In [25]:
knn_params = {"n_neighbors": [2,3,10,20,40,50]}
rf_params = {"max_depth" : [5,8,15,None,10],
             "max_features" : ['auto','sqrt','log2'],
             "min_samples_split" : [2,5,10,15,100],
             "n_estimators": [100,200,300,400,500]}

In [26]:
randomcv_models = [('KNN',KNeighborsRegressor(),knn_params),
                   ('RandomForest',RandomForestRegressor(),rf_params)]

In [31]:
from sklearn.model_selection import RandomizedSearchCV

model_param={}

for name,model,params in randomcv_models:
    random_search = RandomizedSearchCV(model,params,cv=5,n_iter=10,scoring='r2',n_jobs=-1,verbose=2)
    random_search.fit(X_train,y_train)
    
    model_param[name] = random_search.best_params_
    
    
    for model_name in model_param:
        print(f"Best parameters for {model_name} : {model_param[model_name]}")

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best parameters for KNN : {'n_neighbors': 3}
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best parameters for KNN : {'n_neighbors': 3}
Best parameters for RandomForest : {'n_estimators': 300, 'min_samples_split': 2, 'max_features': 'sqrt', 'max_depth': None}


In [40]:
## Retraining the models with best parameters
models = {
    "Random Forest Regressor": RandomForestRegressor(n_estimators=300, min_samples_split=2, max_features='sqrt', max_depth=None, 
                                                     n_jobs=-1),
     "K-Neighbors Regressor": KNeighborsRegressor(n_neighbors=3, n_jobs=-1)
    
}
for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    mse,rmse,mae,r2 = evaluate_model(y_train, y_train_pred)

    mse,rmse,mae,r2 = evaluate_model(y_test,y_test_pred)
    
    print(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(rmse))
    print("- Mean Absolute Error: {:.4f}".format(mae))
    print("- R2 Score: {:.4f}".format(r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(rmse))
    print("- Mean Absolute Error: {:.4f}".format(mae))
    print("- R2 Score: {:.4f}".format(r2))

    print('='*35)
    print('\n')

Random Forest Regressor
Model performance for Training set
- Root Mean Squared Error: 213662.0822
- Mean Absolute Error: 97048.8774
- R2 Score: 0.9394
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 213662.0822
- Mean Absolute Error: 97048.8774
- R2 Score: 0.9394


K-Neighbors Regressor
Model performance for Training set
- Root Mean Squared Error: 321003.4474
- Mean Absolute Error: 125095.5103
- R2 Score: 0.8631
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 321003.4474
- Mean Absolute Error: 125095.5103
- R2 Score: 0.8631


